# Hands-on: Training a basic SST model with Anemoi

In this tutorial we will learn how to train an ML model to predict SST using the anemoi packages. To begin we will look at predicting SST from a 2 degree dataset, using SST at time t as input, to predict SST at time t + 1 day.

We will walk through some of the config choices in anemoi, and then run training on this simple model. There will then be the option to change some key parameters, and see how this impacts scores. By design the model is very simple, with a limited set of inputs and outputs, and small architecture. The focus isn't on predictive skill, but on exploring different design choices and their impact.

**Learning Objectives**
By the end of this tutorial, you will:
- Run a simple ML model to predict ocean SST one day ahead.
- Learn how to configure the anemoi training pipeline and make some simple changes
- Gain understanding of the impact of some key parameters

**Resources**
- [Anemoi Documentation](https://anemoi.readthedocs.io/projects/training/en/latest/)

## Setup

In [ ]:
#ensure in the right dir if not already
current_dir = %pwd
if not (current_dir.endswith("ToySSTProblem") or current_dir.endswith("app")):
    %cd ToySSTProblem
%pwd

We need to ensure we are working in a suitable environment. Depending on your platform, this may already be set up for you (i.e through the docker installation, or directly on EWC). If it isn't, we have provided a requirements.txt file. This is compatible with **python 3.11.10** — other Python versions (e.g. 3.12) may work but are not tested and may encounter package compatibility issues. To set your environment up, first load this version of python. Then install the required libraries with
```
pip install -r requirements.txt
```
Or by updating the condition on the first line to `True` and running the below cell. **Note you then need to restart the kernel**. 

In [ ]:
# Set 'do_install' to False to skip installation (e.g. if already done in a Docker container).
# Set 'do_install' to True to run installation (e.g. if running locally and haven't installed requirements yet).

import subprocess, sys

do_install = True  # Set to True to install/update requirements

if do_install:
    # Remove torch-cluster if present — its CUDA build requires libcudart which
    # may not be on the library path. Anemoi falls back to scikit-learn instead.
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torch-cluster"])

    # Force-reinstall torch from the CUDA 12.4 index. This is necessary because
    # pip considers torch already satisfied if the version number matches, even if
    # a CPU-only build (+cpu) was previously installed by a dependency.
    # nvidia-cudnn-cu12 provides libcudnn.so.9 as a pip package so no system-level
    # cuDNN installation is required.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
        "nvidia-cudnn-cu12",
        "--index-url", "https://download.pytorch.org/whl/cu124",
        "--extra-index-url", "https://pypi.org/simple",
        "--force-reinstall", "--no-deps",
    ])

    # Install all remaining requirements
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

    print('')
    print("*" * 97)
    print("Installation complete. If this is the first time installing, restart the kernel before continuing.")
    print("*" * 97)


In [ ]:
# Import the necessary modules
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings(
    "ignore",
    message="IProgress not found. Please update jupyter and ipywidgets."
)

import torch

import yaml
from omegaconf import OmegaConf
from pathlib import Path

import shutil
import os
import sys
import subprocess
from IPython.display import display, Image

# Load our minimal configuration file
config_path = Path("./training_configs/ToySST.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

import mlflow
import pandas as pd
import matplotlib.pyplot as plt

import zarr, s3fs
from tqdm import tqdm

%matplotlib inline


#### Obtain data

If working on EWC or Edito, you may already have access to the relevant data. Otherwise, a data.zarr file and a tarball with output from a pre-trained model are available on S3 cloud storage. Pull these if required; change the variable NeedData to True and run the below cells.

In [ ]:
## First pull the output from a pre-trained model, and change permissions.
## This should create a directory "pretrained_outputs" containing checkpoints, logs and plots

if Path("pretrained_outputs").exists():
    print("pretrained data exists")
else:
    print("Downloading pretrained outputs")
    !curl -L https://object-store.os-api.cci1.ecmwf.int/ecmwf-training-nwp-ml/training_outputs.tgz | tar xz
    # rename to avoid confusion later
    old = Path("./training_outputs")
    new = Path("./pretrained_outputs")
    shutil.rmtree(new, ignore_errors=True)
    old.rename(new)
    old.mkdir(parents=True, exist_ok=True)
    shutil.copytree(new, old, dirs_exist_ok=True)
    shutil.rmtree(old, ignore_errors=True)

In [ ]:
# Next pull the dataset in zarr format. This will create a file called ToySSTDataset.zarr

class ProgressStore:
    def __init__(self, store, pbar):
        self.store = store
        self.pbar = pbar

    def __getitem__(self, key):
        value = self.store[key]
        self.pbar.update(1)  # update per chunk
        return value

    def keys(self):
        return self.store.keys()

    def __iter__(self):
        return iter(self.store)

    def __len__(self):
        return len(self.store)

    def __contains__(self, key):
        return key in self.store


if Path("ToySSTDataset.zarr").exists():
    print("dataset zarr file exists")
else:
    print("Downloading dataset zarr file")

    s3 = s3fs.S3FileSystem(
        anon=True,
        client_kwargs={"endpoint_url": "https://object-store.os-api.cci1.ecmwf.int"}
    )

    store = s3fs.S3Map(
        root="s3://ecmwf-training-nwp-ml/ToySSTDataset.zarr",
        s3=s3,
        check=False
    )

    local_store = zarr.DirectoryStore("ToySSTDataset.zarr")

    total = len(store)  # number of chunks/files

    with tqdm(total=total, desc="Copying Zarr dataset") as pbar:
        wrapped_store = ProgressStore(store, pbar)
        zarr.copy_store(wrapped_store, local_store, if_exists="replace")

    print("✅ Dataset copied to ToySSTDataset.zarr")

#### Check if using a GPU
The notebook will run on CPU, but will be very slow. If you have access to GPU resource this is highly recommended.

In [ ]:
import subprocess as _sp

print("=== Python & Torch ===")
print(f"Python        : {sys.version.split()[0]}")
print(f"torch version : {torch.__version__}")
print(f"CUDA built    : {torch.version.cuda}  (the CUDA version torch was compiled for)")
print()
print("=== CUDA Runtime ===")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count     : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}       : {torch.cuda.get_device_name(i)}")
else:
    print()
    print("  No GPU detected — possible causes:")
    if "+cpu" in torch.__version__:
        print("  ✗ CPU-only torch build installed (version ends in '+cpu').")
        print("    Fix: re-run the install cell, then restart the kernel.")
    else:
        print("  • torch was built for CUDA — checking driver...")
    _smi = _sp.run(["nvidia-smi"], capture_output=True, text=True)
    if _smi.returncode != 0:
        print("  ✗ nvidia-smi not found or failed — no NVIDIA driver on this machine.")
    else:
        # Extract driver/CUDA line from nvidia-smi header
        for _line in _smi.stdout.splitlines()[:5]:
            if _line.strip():
                print(f"    nvidia-smi: {_line.strip()}")
        print("  • Driver present. The CUDA runtime libraries may be missing.")
        print("    Fix: pip install nvidia-cuda-runtime-cu12, then restart the kernel.")

print()
print("=== nvidia-smi summary ===")
_smi2 = _sp.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                  "--format=csv,noheader"], capture_output=True, text=True)
if _smi2.returncode == 0:
    for _line in _smi2.stdout.strip().splitlines():
        print(" ", _line)
else:
    print("  nvidia-smi not available on this system.")


## Building Our Training Config

We'll work in this notebook, first examining our training configuration, highlighting some key elements. 

The configs set up can be found in the directory training_configs. This is a collection of configuration files defining defaults for various top-level choices, along with a main config file, ToySST.yaml, which defines what is actually used.

We'll load this file and examine it section by section. This configration is combined with the defaults in the training_configs directory, to create the model and training set up. 

### 1: Default Configurations

The top-level decisions around training strategy and model design etc are all set in the defaults at the start of the config file. This is where we make decisions such as which architecture to use, and which training strategy to use (i.e. deterministic prediction which is the default or and ensemble set up, an autoencoder etc). 
- **Model architecture**: gnn, graphtransformer, transformer
- **Graph**: hierarchical, multi scale, ...
- **Training method**: single, ensemble, diffusion, tendency diffusion.

Here we see the defaults chosen for a Toy SST example.


In [ ]:
# Display the default configuration section
print("Default Configurations:")
print("=" * 50)
print(yaml.dump(config['defaults'], default_flow_style=False))

Here we have chosen to use the `zarr` and `native_grid` options for data and dataloader, the standard `evaluation` diagnostics (other options include e.g. ensemble evaluation) a `graph_transformer` architecture, with the `multi_scale` graph, and the `default` training option (this is designed for training a deterministic predictive model).

### Amendments to the default configuration files

Once these broad decisions are defined, more detailed choices are inhereted from the various yaml files within the training_configs directory. Specific changes to these defaults can be listed in the ToySST.yaml file to overwrite the standard options. Here we look at the amendments used for our toy SST example. (Bear in mind the defaults in anemoi relate, in most cases, to medium range weather forecast applications)

### 2: System Configuration

The system configuration (previously reffered to as hardware) defines decisions around the hardware options, such as the number of GPUs being used and how to distrubute these, the inputs (with their full paths) and the location for outputs.

In [ ]:
# Display the hardware configuration section
print("Hardware Configuration:")
print("=" * 50)
print(yaml.dump(config['system'], default_flow_style=False))


### 3: Data Configuration

The data configuration controls how variables are used during training and prediction. It specifies whether each variable is **forcing** (input only), **prognostic** (input and output), or **diagnostic** (output only), and also defines preprocessing/postprocessing choices such as normalisation.

> ⚠️ **Warning:** In this notebook, **forcing** is **not** used in the traditional ocean-modelling sense. Here, it simply means any input-only field, including static fields such as bathymetry or land–sea mask (although these are not used in this toy example).

In [ ]:
# Display the data configuration section
print("Data Configuration:")
print("=" * 50)
print(yaml.dump(config['data'], default_flow_style=False))



The treatment of NaNs is defined -- here by imputing (filling nan values) with 0.

We define a number of variables which indicate the location, time of day and and time of year as forcing, along with 2 metre air temperature (2mT). There are no diagnostic (output only variables). All other variables are automatically defined as prognostic.

The way in which we nornalise data is defined. Setting the default as mean-std (so this is applied to any variables which aren't otherwise listed), and non normalisation ('None') is applied to a few specified variables (chosen because their values already span 0 to 1)

Information about the timestep of the model (i.e. what data spacing to take), the frequency of data available, and the resolution to use is given.

### 4: Dataloader Configuration

The dataloader defines what data to read in from the input files given in system, and how to read this in (batch sizes etc)


In [ ]:
# Display the dataloader configuration section
print("Dataloader Configuration:")
print("=" * 50)
print(yaml.dump(config['dataloader'], default_flow_style=False))


For our toy example, we use batches of size 1.

We read in data from 2020 to 2024. Taking 2t (2m air temperature) from the atmospheric forcings dataset, and taking SST (avg_thetao_1), and various other inputs relating to location, timing etc, from the ocean dataset. 

We limit the model to just 100 batches per epoch (this is for computational speed for this toy example).

The data is split as follows:
- Training period: 2020-2022
- Validation period: 2023
- Test period: 2024

### 5: Graph Configuration

The choice of graph, and whether this is used just for the encoder & decoder, or also used as the processor is defined in the default choices at the top.

Local changes generally include things like the graph shape and resolution

In [ ]:
# Display the graph configuration section
print("Graph Configuration:")
print("=" * 50)
print(yaml.dump(config['graph'], default_flow_style=False))


Here we change the resolution, giving us a fairly coarse grid. 

### 6: Model Configuration

Again the main decisions around what type of model (the architecture etc) are defined by the choice of model in the default configurations.

Local amendments often include the number of channels (i.e. the size and complexity of the model)

In [ ]:
# Display the model configuration section
print("Model Configuration:")
print("=" * 50)
print(yaml.dump(config['model'], default_flow_style=False))


Here we define a relatively small number of channels for this toy problem, otherwise we take the default options for the graphtransformer model.

### 7: Training 

Here we define things about _how_ the model is trained. We configure the training parameters, strategy, and loss function used for training. 

In [ ]:
# Display the training configuration section
print("Training Configuration:")
print("=" * 50)
print(yaml.dump(config['training'], default_flow_style=False))


Here the learning rate starts at `3e-7` and is linearly warmed up over `10` steps to the target learning rate.  After warmup, it follows a **CosineLRScheduler** ([implementation](https://github.com/huggingface/pytorch-image-models/blob/main/timm/scheduler/cosine_lr.py#L19), [ref](https://arxiv.org/pdf/1608.03983)) decay toward the minimum learning rate.

> **Note:** In this setup we run about `50` steps per epoch, so setting `max_steps: 500` is consistent with `max_epochs: 10` (`10 × 50 = 500`). This keeps the two stopping criteria aligned.

Metrics are calculated on the SST variable, avg_thetao_1.

The mean square error (`MSELoss`) loss is used here. We ignore NaNs, so points which were originally NaNs (i.e the land in this ocean example) are not included in the loss. We also scale the loss based on different scalers:
- `node_weights`: Weights each node’s contribution to the loss according to the physical area represented by that graph node, so nodes covering larger regions have a proportionally larger influence on the total loss.
- `general_variable` (not specified): Applies a separate scaling factor to each variable, allowing variables with different magnitudes or priorities to contribute in a balanced way.
- `vertical_level` (not specified): Applies separate scaling factors by vertical level, so errors at different depths can be weighted differently in the total loss.

### 8: Diagnostics Configuration

Here we define what diagnostics will be carried out. Decisions around where to log metrics and what metrics to log are included here.

In [ ]:
# Display the diagnostics configuration section
print("Diagnostics Configuration:")
print("=" * 50)
print(yaml.dump(config['diagnostics'], default_flow_style=False))


The first set of lines describes how to log the data, here we log to mlflow, with an experiment name 'ToySST', and a run name 'testing'. When developing models, multiple runs can be logged under the ToySST experiment name for easy comparison.

We then define the plots to produce. Here we first ask for plots of the loss over training, for the SST parameter (here this is the only predicted variable, but in other applications you can group sets of variables, i.e. all surface variables can form a group). We also ask for plots of sample predictions, again specifying the SST to be the only plotted parameter.

## Training Execution

Now that we have our configuration ready, let's execute the training. We'll run a short training session to verify everything works correctly. First we need to set a couple of environment variables (if running via a scheduler such as slurm these are taken from the slurm directives).

```bash
export ANEMOI_BASE_SEED=42
export ANEMOI_CONFIG_PATH="$PWD/training_configs"
```

We then use the following command to train the model:

```bash
python -m anemoi.training train --config-name ToySST
```

The below cell initiates these commands within this notebook --- if working on the command line, the above would be sufficient.

In [ ]:
cfg_dir = os.getcwd() + "/training_configs"

# Set environment variables
os.environ['ANEMOI_BASE_SEED'] = '42'
os.environ['ANEMOI_CONFIG_PATH'] = cfg_dir
os.environ['POSSIBLE_USER_WARNINGS'] = 'off'
os.environ['TORCH_LOGS'] = "-dynamo,-inductor"
os.environ['HYDRA_FULL_ERROR'] = '1'

# Execute the training using subprocess
print("Starting ML-ocean training...")
print("=" * 50)

process = subprocess.Popen(
    [
        sys.executable, "-m", "anemoi.training", "train",
        "--config-name", "ToySST.yaml"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output
for line in iter(process.stdout.readline, ''):
    if line:
        print(line.rstrip())  # Print immediately to Jupyter cell

# Wait for completion
return_code = process.wait()

print("=" * 50)
if return_code == 0:
    print("✓ Training completed successfully!")
else:
    print(f"❌ Training failed with return code {return_code}")


## Monitoring and Results

### MLflow logging

Several metrics and parameters are logged during training. Here, we access metrics calculated by MLflow during the model training, and use these to assess performance.

We can create plots of key metrics during training, here we look at:
- **Training Loss**: The loss should decrease over time, and indicated if the model is learning to fit the data.
- **Validation Metrics**: Similar to training loss, but calculated on validation data, these results indicate if there is any overfitting.

*Note:* Logging can be done 'online' during training, with results logged to a server and can then be viewed interactively on a website.

Here we set the "Run id" for a pre-trained model (the data we pulled form S3 earlier), to analyse the expected outputs.

First we open the mlflow logs, and print which metrics are available.

In [ ]:
# Set the runId for the run you'd like to investigate.
run_id = "4438386d8b894aa59beb57bb1c7e0b85"

# Connect to the local MLflow tracking server (here a local directory) used by anemoi-training
tracking_uri = f"file://{os.getcwd()}/pretrained_outputs/logs/mlflow"

client = mlflow.tracking.MlflowClient(tracking_uri=tracking_uri)

run = client.get_run(run_id)
print(f"Run name : {run.info.run_name}")
print(f"Run ID   : {run_id}")
print(f"\nAvailable metrics:")
for k in sorted(run.data.metrics.keys()):
    print(f"  {k}")


We then select the training and validation loss per epoch, and produce a line plot showing how these change over the very short training run completed above.

In [ ]:
# Fetch the training and validation loss recorded each epoch
metric_names = ["train_multi_dataset_loss_epoch", "val_multi_dataset_loss_epoch"]

records = []
for metric in metric_names:
    for entry in client.get_metric_history(run_id, metric):
        records.append({"metric": metric, "epoch": entry.step, "loss": entry.value})

df = pd.DataFrame(records)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
for metric, label in zip(metric_names, ["Training loss", "Validation loss"]):
    subset = df[df["metric"] == metric].sort_values("epoch")
    ax.plot(subset["epoch"], subset["loss"], "-o", label=label)

ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()


### Task 1: Change some key training variables and investigate the impact on scores the learning rate, number of channels, and max_epochs

First recreate the plots above with data from your own training runs. You will need to replace the run_id with the value in the output from your training run. This is printed around line 18 of the output, look for `Run id:`.

(Note we have also changed `tracking_uri` to `f"file://{os.getcwd()}/new_training_outputs/logs/mlflow"`, you do not need to change this again if looking at your own runs)


In [ ]:
# Set the runId for the run you'd like to investigate.
run_id = "????"

# Connect to the local MLflow tracking server (here a local directory) used by anemoi-training
tracking_uri = f"file://{os.getcwd()}/new_training_outputs/logs/mlflow"

client = mlflow.tracking.MlflowClient(tracking_uri=tracking_uri)

run = client.get_run(run_id)
print(f"Run name : {run.info.run_name}")
print(f"Run ID   : {run_id}")
print(f"\nAvailable metrics:")
for k in sorted(run.data.metrics.keys()):
    print(f"  {k}")


In [ ]:

# Fetch the training and validation loss recorded each epoch
metric_names = ["train_multi_dataset_loss_epoch", "val_multi_dataset_loss_epoch"]

records = []
for metric in metric_names:
    for entry in client.get_metric_history(run_id, metric):
        records.append({"metric": metric, "epoch": entry.step, "loss": entry.value})

df = pd.DataFrame(records)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
for metric, label in zip(metric_names, ["Training loss", "Validation loss"]):
    subset = df[df["metric"] == metric].sort_values("epoch")
    ax.plot(subset["epoch"], subset["loss"], "-o", label=label)

ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()


Next, adapt the training config (by directly editing the file ToySST.yaml in the training_configs directory) and retrain the model with any of the following (or your own ideas!). You can then plot some of the metrics from these runs by changing the run_id again, as discussed above, and see what impact your changes have on the training and validation loss:
   - Double or half the learning rate (`training: lr: rate`)
   - Provide the model with more data by increasing the batch size (`dataloader: limit_batches: training`)
   - Amend the size of the model, by increasing or decreasing the number of channels (`num_channels`)
   - Train the model for longer by increasing the number of iterations (`max_epochs` and/or `max_steps`)
   - Increase the number of samples considered within each training epoch, by increasing `data_loader: limit_batches: training`

**Questions**
- How does your loss change with these adaptions? 
- How much longer does the training take (per iteration and/or in total)?
- Does the relationshop between training and validation loss change? Are there any signs of overfitting? (you may need to carry out longer runs to be able to assess this)

***Note***:

You do not need to do the training in the notebook. Once the ToySST.yaml file has been edited, you can either rerun the training cell of the notebook, or you can run directly on the console with: 
```
anemoi-training train --config-path=PATH-TO-YOUR-CONFIG-FOLDER --config-name=ToySST
```

#### For a slightly more complex change
   - Remove the 2m temperature forcing field from the model (hint, you will need to consider both the `data` and `dataloader` parts of the configuration).
   How does this impact the training and validation scores?


### Inference of your model

Training is about letting the model learn the optimal set of weights to make predictions. Once these are learned, we can use the model to make predictions, this is referred to as inference.

To run inference on a pretrained model based on the above, go to the Jupyter notebook `inference_ToySST.ipynb`. Here you can also test running inference on the models you've trained yourselves.

